In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://internetlab.org.br/ajax/"
SITE_URL = "https://internetlab.org.br"
print("✅ Scraper do InternetLab Blog pronto!")

✅ Scraper do InternetLab Blog pronto!


In [3]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [4]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [5]:
# %%
def load_articles_from_db():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT *
        FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles_from_db()
display(df_db.head())
print(f"📦 Total no banco: {len(df_db)} registros")


,id,title,date,author,url,source
0,792,Nueva sentencia de la Corte Europea determina ...,"9 septiembre, 2021",Observacom,https://www.observacom.org/nueva-sentencia-de-...,Observacom
1,1100,Gigantes tecnológicos centralizan el poder de ...,"9 septiembre, 2020",Observacom,https://www.observacom.org/gigantes-tecnologic...,Observacom
2,1356,Lobby de grupos evangélicos para reformar ley ...,"9 septiembre, 2019",Observacom,https://www.observacom.org/lobby-de-grupos-eva...,Observacom
3,1960,AMARC señala falta de voluntad política para r...,"9 septiembre, 2016",Observacom,https://www.observacom.org/amarc-senala-falta-...,Observacom
4,236,"Meta, Google y TikTok concentran el 70% del tr...","9 octubre, 2024",Observacom,https://www.observacom.org/meta-google-y-tikto...,Observacom


📦 Total no banco: 3047 registros


In [6]:
# %%
def montar_url(pagina):
    """Monta a URL da API AJAX do InternetLab para a página indicada."""
    return f"{BASE_URL}?call=blog&lang=pt&page={pagina}"

def extrair_paragrafos(url):
    """Acessa a página do post e extrai os parágrafos principais."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        ps = soup.find_all("p")
        textos = [
            p.get_text(strip=True)
            for p in ps
            if len(p.get_text(strip=True).split()) > 10
        ]
        return textos[:5] if textos else ["NA"]
    except:
        return ["NA"]

In [ ]:
# %%
noticias = []
TOTAL_PAGES = 150  # páginas suficientes para cobrir todo o histórico, vcs podem ajustar conforme o necessário, ok?

for pagina in range(1, TOTAL_PAGES + 1):
    url = montar_url(pagina)
    print(f"📄 Coletando página {pagina}: {url}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"⚠️ Erro HTTP {r.status_code} — parando.")
            break

        data = r.json()
    except Exception as e:
        print(f"⚠️ Erro ao acessar a API: {e}")
        break

    if not data.get("status") or not data.get("answer"):
        print("🏁 Sem mais resultados — fim da coleta.")
        break

    posts = data["answer"]
    print(f"   {len(posts)} posts encontrados")

    for post in posts:
        titulo = post.get("title", "")
        link = post.get("url", "")
        data_fmt = post.get("date_formatted", "NA")
        categoria = post.get("category", "")
        area = post.get("areas_pesquisa", "")
        autores = post.get("autores", "InternetLab")

        # extrai parágrafos da página do post
        paragrafos = extrair_paragrafos(link)

        # acumula
        noticias.append({
            "titulo": titulo,
            "data": data_fmt,
            "link": link,
            "categoria": categoria,
            "area": area,
            "autores": autores,
            "paragrafos": " || ".join(paragrafos),
            "fonte": "InternetLab"
        })

        # grava no banco
        insert_article(
            title=titulo,
            date=data_fmt,
            author=autores,
            url=link,
            source="InternetLab"
        )

    time.sleep(1)

print(f"\n✅ Total coletado: {len(noticias)} posts")

df_internetlab = pd.DataFrame(noticias)
display(df_internetlab.head())

📄 Coletando página 1: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=1
   6 posts encontrados
📄 Coletando página 2: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=2
   6 posts encontrados
📄 Coletando página 3: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=3
   6 posts encontrados
📄 Coletando página 4: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=4
   6 posts encontrados
📄 Coletando página 5: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=5
   6 posts encontrados
📄 Coletando página 6: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=6
   6 posts encontrados
📄 Coletando página 7: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=7
   6 posts encontrados


In [ ]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

In [ ]:
keywords = ['digital', 'internet', 'IA', 'tecnologia', 'dados', 'privacidade']
pattern = r'|'.join(keywords)

df_filt = df_internetlab[
    df_internetlab['titulo'].str.contains(pattern, case=False, na=False, regex=True) |
    df_internetlab['paragrafos'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print(f"{len(df_filt)} posts filtrados (de {len(df_internetlab)})")
display(df_filt.head())

In [ ]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para gráficos")
        return

    # ------------------------------
    # Top 15
    # ------------------------------
    top15 = df.head(15).copy()
    top15['rank'] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x='rank',
        y='title',
        orientation='h',
        title='Top 15 Notícias – Internet Governance'
    )
    fig1.update_layout(height=600)
    fig1.show()

    # ------------------------------
    # Pizza por Fonte (BANCO)
    # ------------------------------
    # Fonte
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # ------------------------------
    # Nuvem de Palavras
    # ------------------------------
    text = ' '.join(df['title'].astype(str)).lower()
    words = re.findall(r'\b\w{4,}\b', text)

    wc = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    wc.columns = ['palavra', 'freq']

    fig3 = px.treemap(
        wc,
        path=['palavra'],
        values='freq',
        title='Nuvem de Palavras – Títulos'
    )
    fig3.show()

In [ ]:
plot_charts(df_db)